# INTRODUCTION

Notre projet implémente une architecture Data Lakehouse complète, qui combine la flexibilité d'un Data Lake avec la structure d'un Data Warehouse. Nous avons utilisé  Apache Spark pour le traitement, Apache Kafka pour le temps réel, MinIO comme stockage objet (compatible S3), et PostgreSQL pour le serving final. L'architecture Data Lakehouse suit le modèle médaillon en trois couches : Bronze, Silver et Gold.

## Imports et session Spark

In [1]:
import os    #  pour manipuler les variables d'environnement et les chemins de fichiers

# Définit le répertoire Windows où sont stockés tous les fichiers JAR nécessaires à Spark.
JARS_DIR = r"C:\spark-jars"


''' 
Liste les 6 JARs requis :

 - spark-sql-kafka : connecteur Spark ↔ Kafka
 - spark-token-provider-kafka : gestion des tokens d'auth Kafka
 - kafka-clients : bibliothèque client Kafka Java
 - commons-pool2 : pool de connexions (dépendance Kafka)
 - hadoop-aws : connecteur Hadoop ↔ S3/MinIO via protocole S3A
 - aws-java-sdk-bundle : SDK AWS complet pour MinIO 
'''

jar_files = [
    rf"{JARS_DIR}\spark-sql-kafka-0-10_2.12-3.5.0.jar",
    rf"{JARS_DIR}\spark-token-provider-kafka-0-10_2.12-3.5.0.jar",
    rf"{JARS_DIR}\kafka-clients-3.5.1.jar",
    rf"{JARS_DIR}\commons-pool2-2.11.1.jar",
    rf"{JARS_DIR}\hadoop-aws-3.3.4.jar",
    rf"{JARS_DIR}\aws-java-sdk-bundle-1.12.262.jar",
]


'''
Vérifie l'existence physique de chaque JAR sur le disque. 
Si un JAR est absent, lève une exception immédiatement pour 
éviter un crash silencieux plus tard.
'''
print("Vérification JARs :")
all_ok = True
for j in jar_files:
    ok = os.path.exists(j)
    if not ok:
        all_ok = False
    print(f"  {'✔' if ok else ' MANQUANT'} {os.path.basename(j)}")

if not all_ok:
    raise FileNotFoundError("Des JARs manquent dans C:\\spark-jars")

jars_csv      = ",".join(jar_files)
classpath_str = ";".join(jar_files)


'''
Cette variable doit être définie avant tout import de PySpark.
Elle passe les JARs au JVM driver de Spark dès son démarrage. 
8g alloue 8 Go de RAM au driver.
'''

os.environ["PYSPARK_SUBMIT_ARGS"] = (
    f"--driver-memory 8g "
    f"--driver-class-path \"{classpath_str}\" "
    f"--jars \"{jars_csv}\" "
    f"pyspark-shell"
)
print("\n PYSPARK_SUBMIT_ARGS prêt")

#  JAVA / HADOOP 
os.environ["JAVA_HOME"]   = r"C:\Program Files\Java\jdk-17"
os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["PATH"]        = r"C:\hadoop\bin;" + os.environ.get("PATH", "")

'''
Définit les variables d'environnement Java et Hadoop. Hadoop doit être 
dans le PATH pour que Spark puisse trouver winutils.exe sous Windows.
'''
os.makedirs(r"C:\hadoop\bin", exist_ok=True)
if not os.path.exists(r"C:\hadoop\bin\winutils.exe"):
    with open(r"C:\hadoop\bin\winutils.exe", "wb") as f:
        f.write(b"MZ")



# ---------  Imports et création de la session Spark  ------------------------------------


import pandas as pd
import boto3     #  SDK Python pour communiquer avec MinIO
import pyarrow.parquet as pq    #   lecture de fichiers Parquet depuis MinIO
import io, csv, random
from datetime import datetime, timedelta  # seront utilisés plus tard pour la génération des logs.
from sqlalchemy import create_engine, text    #connexion et chargement vers PostgreSQL
import psycopg2
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
import json
import time
from kafka import KafkaProducer
from pyspark.sql.functions import *
from pyspark.sql.types import *
import random
import pandas as pd
from datetime import datetime, timedelta
from pyspark.sql import Window



# Convertit les chemins Windows (\) en URI file:/// avec des /  format requis par la configuration Spark.
jars_str = ",".join(f"file:///{p.replace(chr(92), '/')}" for p in jar_files)



'''
Crée la session Spark unique et les configurations importantes :

 - local[*] : utilise tous les cœurs disponibles sur la machine
 - shuffle.partitions = 50 : réduit le nombre de partitions par défaut (200) pour un environnement local
 - memory.fraction = 0.8 : 80% de la heap JVM est dédiée à Spark.

 Configuration du connecteur S3A vers MinIO local
'''

spark = (
    SparkSession.builder
    .appName("DataLakehouse")
    .master("local[*]")
    .config("spark.driver.memory",             "8g")
    .config("spark.driver.maxResultSize",      "4g")
    .config("spark.sql.shuffle.partitions",    "50")
    .config("spark.memory.fraction",           "0.8")
    .config("spark.hadoop.fs.s3a.endpoint",               "http://localhost:9000")
    .config("spark.hadoop.fs.s3a.access.key",             "minioadmin")
    .config("spark.hadoop.fs.s3a.secret.key",             "minioadmin")
    .config("spark.hadoop.fs.s3a.path.style.access",      "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.impl",                   "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .getOrCreate()
)

print(" Spark", spark.version, "démarré")
jars = spark.sparkContext._jsc.sc().listJars()
print(f"   {jars.size()} JARs chargés")

Vérification JARs :
  ✔ spark-sql-kafka-0-10_2.12-3.5.0.jar
  ✔ spark-token-provider-kafka-0-10_2.12-3.5.0.jar
  ✔ kafka-clients-3.5.1.jar
  ✔ commons-pool2-2.11.1.jar
  ✔ hadoop-aws-3.3.4.jar
  ✔ aws-java-sdk-bundle-1.12.262.jar

 PYSPARK_SUBMIT_ARGS prêt
 Spark 3.5.8 démarré
   6 JARs chargés


In [2]:
# Test de connexion Kafka : Simple test de connectivité : tente d'instancier un producteur Kafka sur le broker local. Si Kafka n'est pas démarré, une exception est levée ici.

from kafka import KafkaProducer

KafkaProducer(bootstrap_servers="127.0.0.1:9092")

In [18]:
#  Test de l'API AviationStack  : envoie une requête à l'API AviationStack pour vérifier que la clé API est valide et que la connexion fonctionne.

import requests

ACCESS_KEY = "d01623d5beaac50877e71d659ca0b221"

url = "http://api.aviationstack.com/v1/flights"

params = {
    "access_key": ACCESS_KEY,
    "flight_status": "active"
}

r = requests.get(url, params=params)

if r.status_code == 200 and "data" in r.json():
    print(" La connexion fonctionne — clé API valide.")
else:
    print(" Échec de la connexion :", r.json())

 La connexion fonctionne — clé API valide.


---
## Traitement par lots (Batch)
---

## Couche Bronze (vérification d'ingestion brute)

In [ ]:
# Lire flights.csv (données des vols (données 2024 le mois 1) US)
flights_df = spark.read \
    .option("header", True) \
    .csv("s3a://data-lakehouse1/bronze/flights.csv")

# Lire airports.csv (informations descriptives des aéroports)
airports_df = spark.read \
    .option("header", True) \
    .csv("s3a://data-lakehouse1/bronze/airports.csv")

# weather.json (données météo) , option "multiLine=True" car le JSON est sur plusieurs lignes
weather_df = spark.read \
    .option("multiLine", True) \
    .json("s3a://data-lakehouse1/bronze/weather.json")


#  Affiche un aperçu et compte les lignes et des colonnes pour validation

print("flights.csv :")
flights_df.show(2)
print("  lignes   :", flights_df.count())
print("  colonnes :", len(flights_df.columns))

print("airports.csv :")
airports_df.show(2)
print("  lignes   :", airports_df.count())
print("  colonnes :", len(airports_df.columns))

print("weather.json :")
weather_df.show(2)
print("  lignes   :", weather_df.count())
print("  colonnes :", len(weather_df.columns))

flights.csv :
+----+-------+-----+----------+---------+----------+-------------------------+---------------------------------------+------------------------+---------------------------+-------------------------------+---------------------------------------+----------------------------------------------+-------------------------------------------------+--------------------------------------------------+------------------+------------------------+---------------------------+-----------+-------------------------------+---------------+------------------+------------------+------+--------------+-----------+---------------+---------------+---------+-------------+----------------+----------------+----+------------+---------+-------------+-------------+-------+----------+-------+--------+---------------+--------+--------------------+----------+-------+---------+--------+------+----------+-------+--------+---------------+--------+------------------+----------+---------+----------------+--------

In [4]:
# Ajout de ingestion_date sur chaque dataset 


'''
L'ajout de  "inferSchema=True" pour 'flights_df' (détection automatique des types) et
 ajoute une colonne 'ingestion_date' avec l'horodatage exact de lecture.
 C'est une métadonnée de traçabilité de la couche Bronze.
'''

# flights.csv
flights_df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("s3a://data-lakehouse1/bronze/flights.csv") \
    .withColumn("ingestion_date", current_timestamp())

# airports.csv
airports_df = spark.read \
    .option("header", True) \
    .csv("s3a://data-lakehouse1/bronze/airports.csv") \
    .withColumn("ingestion_date", current_timestamp())

# weather.json
weather_df = spark.read \
    .option("multiLine", True) \
    .json("s3a://data-lakehouse1/bronze/weather.json") \
    .withColumn("ingestion_date", current_timestamp())


print("  flights.csv + ingestion_date")
flights_df.show(3)

print("  airports.csv + ingestion_date")
airports_df.show(3)

print("  weather.json + ingestion_date")
weather_df.show(3)

  flights.csv + ingestion_date
+----+-------+-----+----------+---------+----------+-------------------------+---------------------------------------+------------------------+---------------------------+-------------------------------+---------------------------------------+----------------------------------------------+-------------------------------------------------+--------------------------------------------------+------------------+------------------------+---------------------------+-----------+-------------------------------+---------------+------------------+------------------+------+--------------+-----------+---------------+---------------+---------+-------------+----------------+----------------+----+--------------+---------+-------------+-------------+-------+----------+-------+--------+---------------+--------+--------------------+----------+-------+---------+--------+------+----------+-------+--------+---------------+--------+------------------+----------+---------+------

## Couche Silver (nettoyage + enrichissement)

### nettoyage des data sats  

In [6]:

# ------------------------------------------------------------
#   Nettoyage de flights (vols)
# ------------------------------------------------------------

'''
 - Supprime les doublons.
 - Remplace les valeurs NULL des colonnes numériques critiques par 0.
 - Cast explicite en IntegerType ( car CSV lit tout en String).
 - Convertit la colonne date de String en type DateType selon le format "yyyy-MM-dd".
 - 
'''

flights_clean = (
    flights_df
    .dropDuplicates(["FlightDate", "Flight_Number_Marketing_Airline", "OriginAirportID", "DestAirportID"])
    .fillna({"Cancelled": 0, "DepDelayMinutes": 0, "ArrDelayMinutes": 0})
    .withColumn("DepDelayMinutes", col("DepDelayMinutes").cast(IntegerType()))
    .withColumn("ArrDelayMinutes", col("ArrDelayMinutes").cast(IntegerType()))
    .withColumn("Cancelled", col("Cancelled").cast(IntegerType()))
    .withColumn("OriginAirportID", col("OriginAirportID").cast(IntegerType()))
    .withColumn("DestAirportID", col("DestAirportID").cast(IntegerType()))
    .withColumn("FlightDate", to_date(col("FlightDate"), "yyyy-MM-dd"))
)


# ------------------------------------------------------------
#   Nettoyage de airports (aéroports)
# ------------------------------------------------------------

'''
 - dropDuplicates(["id"]) : dédoublonnage sur la clé primaire
 - filter(col("id") != 0) : supprime les aéroports avec id=0 (invalides)
 - trim : supprime les espaces autour du code IATA
'''


airports_clean = (
    airports_df
    .dropDuplicates(["id"])
    .withColumn("id", col("id").cast(IntegerType()))
    .filter(col("id").isNotNull())
    .filter(col("id") != 0)
    .filter(col("iso_country") == "US")
    .withColumn("iata_code", trim(col("iata_code")))
    .fillna({"iata_code": "Unknown" })
    .select(
        "id",
        "iata_code",
        "name",
        "municipality",
        "iso_country",
        "latitude_deg",
        "longitude_deg"
    )
)

# les plus part des aeroports sont "iata_code" est null

# ------------------------------------------------------------
#   Nettoyage weather (météo)
# ------------------------------------------------------------

'''
 - Dédoublonnage par date+ville 
 - valeurs nulles remplacées par 0
 - filtre les lignes sans condition météo
 - renommage des colonnes en conventions plus claires
'''

weather_clean = (
    weather_df
    .dropDuplicates(["date", "city"])
    .fillna({
        "precipitation_mm": 0.0,
        "Temperature": 0.0,
        "wind_speed_kmh": 0.0
    })
    .filter(col("condition").isNotNull())
    .withColumn("date", to_date(col("date"), "yyyy-MM-dd"))
    .withColumnRenamed("date", "Date")
    .withColumnRenamed("Temperature", "Temperature_C")
    .withColumnRenamed("wind_speed_kmh", "Wind_Speed_km_h")
    .withColumnRenamed("precipitation_mm", "Precipitation_mm")
    .withColumnRenamed("condition", "Condition")
    .withColumnRenamed("city", "City")
)

### Jointure des data sets  

In [7]:
# ------------------------------------------------------------
#  Join de flights avec airports
# ------------------------------------------------------------

'''
Crée un broadcast du référentiel aéroports pour l'aéroport d'origine.
Le broadcast force Spark à envoyer ce petit DataFrame à tous les executors,
évitant un shuffle coûteux lors de la jointure.
'''

origin_lookup = broadcast(
    airports_clean.select(
        col("id").alias("origin_airport_id"),
        col("iata_code").alias("origin_iata_code"),
        col("name").alias("origin_airport_name"),
        col("municipality").alias("origin_city"),
        col("iso_country").alias("origin_country")
    )
)

# Jointure des vols avec les aéroports  sur id
flights_with_origin = flights_clean.join(
    origin_lookup,
    flights_clean["OriginAirportID"] == origin_lookup["origin_airport_id"],
    "left"
)

dest_lookup = broadcast(
    airports_clean.select(
        col("id").alias("dest_airport_id"),
        col("iata_code").alias("dest_iata_code"),
        col("name").alias("dest_airport_name"),
        col("municipality").alias("dest_city"),
        col("iso_country").alias("dest_country")
    )
)

flights_with_airport = flights_with_origin.join(
    dest_lookup,
    flights_with_origin["DestAirportID"] == dest_lookup["dest_airport_id"],
    "left"
)

# ------------------------------------------------------------
#  Join de flights_airport avec weather
# ------------------------------------------------------------


'''
Jointure des vols avec la météo sur la date. Le 'drop("Date")' évite d'avoir deux colonnes de date en doublon après la jointure.
'''
flights_enriched = flights_with_airport.join(
    broadcast(weather_clean),
    flights_with_airport["FlightDate"] == weather_clean["Date"],
    "left"
).drop("Date")


if "ingestion_date" in flights_enriched.columns:
    flights_enriched = flights_enriched.drop("ingestion_date")

flights_enriched = flights_enriched.withColumn(
    "ingestion_date",
    current_timestamp()
)

print("✔ Jointure OK")
print("  lignes   :", flights_enriched.count())
print("  colonnes :", len(flights_enriched.columns))

# Sauvegarde Silver batch (2024)

'''
Sauvegarde le résultat enrichi en Parquet (format columaire compressé) dans MinIO, couche Silver. 
mode("overwrite") écrase les données existantes.
'''

flights_enriched.write \
    .mode("overwrite") \
    .parquet("s3a://data-lakehouse1/silver/flights_enriched/")

print("✔ Sauvegarde vers MinIO OK")

✔ Jointure OK
  lignes   : 28523390
  colonnes : 137
✔ Sauvegarde vers MinIO OK


---
## Traitement en temps réel (Strem)
---

## Couche Bronze (ingestion Temps réel)

### Producteur Kafka (AviationStack  → Kafka)

In [8]:

# Configuration du producteur Kafka

'''
Crée le producteur Kafka avec :

value_serializer : sérialise chaque message Python dict → JSON → bytes
acks=1 : le broker confirme la réception sans attendre la réplication
retries=5 : retry automatique en cas d'échec temporaire
api_version=(2, 6, 0) : force IPv4, évite les problèmes de résolution IPv6 sous Windows
'''
producer = KafkaProducer(
    bootstrap_servers="127.0.0.1:9092",
    value_serializer=lambda v: json.dumps(v).encode("utf-8"),
    acks=1,
    retries=5,
    linger_ms=0,
    request_timeout_ms=30000,
    api_version=(2, 6, 0)      # ← force IPv4, évite la résolution IPv6
)

# On garde le même topic pour ton pipeline Spark
TOPIC = "flightaware-flights"

# Configuration de l'API de AviationStack
ACCESS_KEY = "d01623d5beaac50877e71d659ca0b221"
URL = "http://api.aviationstack.com/v1/flights"

# Paramètres de requêtage pour AviationStack (vols actifs en direct)
PARAMS = {
    "access_key": ACCESS_KEY,
    "flight_status": "active"
}


'''
Boucle pendant 10 minutes. Récupère au maximum 20 vols actifs par appel (limite du plan gratuit AviationStack).
'''
def run(duration_sec=600):
    start = time.time()

    while time.time() - start < duration_sec:
        try:
            # Appel API AviationStack avec les paramètres de requêtage URL
            r = requests.get(URL, params=PARAMS, timeout=20)
            
            if r.status_code == 200:
                data = r.json()
                # Extraction de la liste des vols (clé 'data' chez AviationStack)
                flights = data.get("data", [])[:20]

                for flight in flights:
                    # Extraction sécurisée des sous-blocs imbriqués d'AviationStack
                    departure = flight.get("departure", {})
                    arrival = flight.get("arrival", {})
                    airline = flight.get("airline", {})
                    aircraft = flight.get("aircraft", {})
                    flight_info = flight.get("flight", {})

                    # Mappe les champs imbriqués de l'API AviationStack vers une structure plate, puis envoie le message dans le topic Kafka flightaware-flights.

                    msg = {
                        "fa_flight_id": f"{airline.get('icao')}-{flight_info.get('number')}-{flight.get('flight_date')}"if airline.get('icao') else None,

                        "icao24": aircraft.get("icao24"),
                        "ident": flight_info.get("icao"),                                               # Équivalent de callsign (ex: DHK1914)
                        "origin": departure.get("icao"),                                                # Code ICAO de l'aéroport d'origine (ex: KLAX)
                        "destination": arrival.get("icao"),                                             # Code ICAO de l'aéroport de destination (ex: KPHX)
                        "status": flight.get("flight_status"),
                        "departure_delay": departure.get("delay"),                                      # Fourni en minutes par AviationStack (ou None)
                        "arrival_delay": arrival.get("delay"),                                          # Fourni en minutes par AviationStack (ou None)
                        "scheduled_out": departure.get("scheduled"),                                    # Timestamp ISO décollage théorique
                        "actual_out": departure.get("actual"),                                          # Timestamp ISO décollage réel
                        "aircraft_type": aircraft.get("icao"),                                          # Type d'avion au format ICAO (ex: B763)
                        "fetch_timestamp": int(time.time())
                    }
                    producer.send(TOPIC, msg)

                producer.flush()
                print("✔ batch AviationStack envoyé à Kafka:", len(flights))
            else:
                print(f"Erreur API AviationStack (Status {r.status_code}):", r.text)

        except Exception as e:
            print("Erreur script Producteur:", e)

        # Pause de 60 secondes entre chaque appel pour respecter le quota mensuel du plan gratuit.
        time.sleep(60)   


    print("✔ arrêt après 10 minutes")

if __name__ == "__main__":
    run()

✔ batch AviationStack envoyé à Kafka: 20
✔ batch AviationStack envoyé à Kafka: 20
✔ batch AviationStack envoyé à Kafka: 20
✔ batch AviationStack envoyé à Kafka: 20
✔ batch AviationStack envoyé à Kafka: 20
✔ batch AviationStack envoyé à Kafka: 20
✔ batch AviationStack envoyé à Kafka: 20
✔ batch AviationStack envoyé à Kafka: 20
✔ batch AviationStack envoyé à Kafka: 20
✔ batch AviationStack envoyé à Kafka: 20
✔ arrêt après 10 minutes


### Spark Streaming 

In [9]:
# ─────────────────────────────────────────────
# 1. LECTURE 
# ─────────────────────────────────────────────

# crée un DataFrame de streaming connecté à Kafka
# startingOffsets="earliest" relit depuis le début du topic. failOnDataLoss=false évite l'arrêt du stream si des offsets sont manquants.

kafka_df = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "127.0.0.1:9092")
    .option("subscribe", "flightaware-flights") 
    .option("startingOffsets", "earliest")
    .option("failOnDataLoss", "false")
    .load()
)

# ─────────────────────────────────────────────
# 2. SCHEMA D'EXTRACTION ET INTERPRETATION DES MESSAGES KAFKA
# ─────────────────────────────────────────────

# Définit le schéma attendu pour l'analyse du JSON. Sans schéma explicite, Spark ne peut pas désérialiser le JSON.
flightaware_schema = StructType([
    StructField("fa_flight_id",     StringType()),
    StructField("icao24",           StringType()),
    StructField("ident",            StringType()),
    StructField("origin",           StringType()),
    StructField("destination",      StringType()),
    StructField("status",           StringType()),
    StructField("departure_delay",  DoubleType()), 
    StructField("arrival_delay",    DoubleType()), 
    StructField("scheduled_out",    StringType()),
    StructField("actual_out",       StringType()),
    StructField("aircraft_type",    StringType()),
    StructField("fetch_timestamp",  LongType())
])

# ─────────────────────────────────────────────
# 3. ANALYSE DU JSON 
# ─────────────────────────────────────────────
flightaware = (
    kafka_df
    .selectExpr("CAST(value AS STRING) as json")
    .withColumn("data", from_json(col("json"), flightaware_schema))
    .select("data.*")
    # AviationStack fournit les retards en minutes, on les convertit en secondes pour la couche Bronze
    .withColumn("departure_delay", col("departure_delay") * 60)
    .withColumn("arrival_delay", col("arrival_delay") * 60)
    # Heure d'ingestion dans la couche Bronze
    .withColumn("ingestion_date", current_timestamp())
)

# ─────────────────────────────────────────────
# 4. ÉCRITURE STREAMING VERS MINIO 
# ─────────────────────────────────────────────
'''
Écrit le stream en continu vers MinIO en CSV. Le checkpointLocation sauvegarde l'état du stream (offset Kafka consommé) 
pour reprendre sans perte en cas de crash.
'''

# Spark ajoute les fichiers au fur et à mesure ("append") dans le dossier spécifié.

bronze_query = (
    flightaware.writeStream
    .format("csv")
    .option("header", "true")
    .option("path", "s3a://data-lakehouse1/bronze/flightaware_radar/")
    # Le checkpoint stocke l'état du flux pour savoir où Spark s'est arrêté en cas de crash
    .option("checkpointLocation", "s3a://data-lakehouse1/checkpoints/flightaware_radar/")
    .outputMode("append")
    .start()
)

print("✔ Flux d'ingestion Streaming vers MinIO initialisé avec succès ")

✔ Flux d'ingestion Streaming vers MinIO initialisé avec succès 


---

## Couche Silver — Nettoyage des donnes en temps reel

In [10]:

# ─────────────────────────────────────────────
# SCHEMA BRONZE STRICTEMENT ALIGNÉ SUR LE FLUX STREAMING
# ─────────────────────────────────────────────

#  Le schéma de lecture Bronze utilise StringType pour tous les champs car les CSV écrits par Spark Streaming ne préservent pas les types.

flightaware_bronze_schema = StructType([
    StructField("fa_flight_id",     StringType(), True),
    StructField("icao24",           StringType(), True),
    StructField("ident",            StringType(), True),   
    StructField("origin",           StringType(), True),   
    StructField("destination",      StringType(), True),   
    StructField("status",           StringType(), True),
    StructField("departure_delay",  StringType(), True),   
    StructField("arrival_delay",    StringType(), True),   
    StructField("scheduled_out",    StringType(), True),   
    StructField("actual_out",       StringType(), True),   
    StructField("aircraft_type",    StringType(), True),
    StructField("fetch_timestamp",  StringType(), True),
    StructField("ingestion_time",   StringType(), True)    
])

# Définition obligatoire des deux chemins MinIO
BRONZE_PATH = "s3a://data-lakehouse1/bronze/flightaware_radar/"
SILVER_PATH = "s3a://data-lakehouse1/silver/flightaware_radar_nettoye/"

# ─────────────────────────────────────────────
# LECTURE BRONZE SÉCURISÉE 
# ─────────────────────────────────────────────

# si le dossier Bronze n'existe pas encore (stream pas encore démarré), on crée un DataFrame vide plutôt que de crasher.
try:
    flightaware_bronze = (
        spark.read
        .option("header", "false") 
        .schema(flightaware_bronze_schema)
        .csv(BRONZE_PATH)
    )
    # On vérifie si le DataFrame contient de la donnée sans déclencher de lourde action
    path_exists = True if flightaware_bronze.take(1) else False
except Exception as e:
    # Si le dossier n'existe pas encore sur MinIO, Spark lève une exception interceptée ici
    print(f"⚠ Le dossier Bronze n'existe pas encore ou est vide à l'adresse : {BRONZE_PATH}")
    path_exists = False
    flightaware_bronze = spark.createDataFrame(spark.sparkContext.emptyRDD(), flightaware_bronze_schema)

# Filtre les lignes d'en-tête parasites qui peuvent apparaître quand Spark Streaming écrit plusieurs micro-batches CSV avec des headers répétés.
flightaware_bronze = flightaware_bronze.filter(col("fa_flight_id") != "fa_flight_id")


# ─────────────────────────────────────────────
# NETTOYAGE SILVER
# ─────────────────────────────────────────────


'''
Nettoyage Silver :

 - Dédoublonnage sur fa_flight_id + fetch_timestamp
 - Conversion String→Double pour les retards
 - Parsing des timestamps ISO en TimestampType
 - Extraction de la date de vol depuis scheduled_out
 - Reconversion des secondes en minutes avec coalesce(..., 0.0) (NULL → 0)
 - Détection d'annulation via le champ status (texte libre)
'''
flightaware_radar_clean = (
    flightaware_bronze
    # Dédoublonnage sur identifiant de vol + timestamp
    .dropDuplicates(["fa_flight_id", "fetch_timestamp"])
    
    # Conversion sécurisée des retards String -> Double
    .withColumn("departure_delay_s", col("departure_delay").cast(DoubleType()))
    .withColumn("arrival_delay_s",   col("arrival_delay").cast(DoubleType()))
    
    # Conversion des timestamps ISO en TimestampType
    .withColumn("scheduled_out_ts", to_timestamp(col("scheduled_out")))
    .withColumn("actual_out_ts",    to_timestamp(col("actual_out")))
    
    # Date de vol extraite du décollage prévu
    .withColumn("flight_date", to_date(col("scheduled_out")))
    # Filtrer les lignes sans date valide pour éviter de corrompre le partitionnement futur
    .filter(col("flight_date").isNotNull())
    
    # Conversion des secondes en minutes avec gestion des valeurs NULL (coalesce)
    .withColumn(
        "departure_delay_min",
        coalesce(round(col("departure_delay_s") / 60, 1), lit(0.0))
    )
    .withColumn(
        "arrival_delay_min",
        coalesce(round(col("arrival_delay_s") / 60, 1), lit(0.0))
    )
    
    # Sécurisation du calcul d'annulation : gestion du cas où status est NULL
    .withColumn(
        "cancelled",
        when(col("status").isNotNull() & lower(col("status")).contains("cancelled"), 1)
        .otherwise(0)
        .cast(IntegerType())
    )
    
    # Nettoyage callsign et ajout de la date d'ingestion Silver
    .withColumn("callsign", trim(col("ident")))
    .withColumn("ingestion_date", current_timestamp())
    
    # Sélection finale des colonnes pour la couche Silver
    .select(
        "fa_flight_id",
        "icao24",
        "callsign",        
        "origin",            
        "destination",       
        "status",
        "flight_date",       
        "scheduled_out_ts",
        "actual_out_ts",
        "departure_delay_min",
        "arrival_delay_min",
        "cancelled",
        "aircraft_type",
        "ingestion_date"
    )
    .cache()
)

#print("Lignes Silver FlightAware/AviationStack :", traject_radar_clean.count())

# ─────────────────────────────────────────────
# ECRITURE SILVER (En Parquet)
# ─────────────────────────────────────────────
if path_exists and flightaware_radar_clean.count() > 0:
    flightaware_radar_clean.write \
        .mode("overwrite") \
        .parquet(SILVER_PATH)
    print("✔ Sauvegarde Silver vers MinIO OK :", SILVER_PATH)
    flightaware_radar_clean.show(5, truncate=False)
else:
    print("✖ Aucune donnée valide à écrire dans la couche Silver (Dossier absent ou vide).")

✔ Sauvegarde Silver vers MinIO OK : s3a://data-lakehouse1/silver/flightaware_radar_nettoye/
+-------------------+------+--------+------+-----------+------+-----------+-------------------+-------------------+-------------------+-----------------+---------+-------------+--------------------------+
|fa_flight_id       |icao24|callsign|origin|destination|status|flight_date|scheduled_out_ts   |actual_out_ts      |departure_delay_min|arrival_delay_min|cancelled|aircraft_type|ingestion_date            |
+-------------------+------+--------+------+-----------+------+-----------+-------------------+-------------------+-------------------+-----------------+---------+-------------+--------------------------+
|CNV-None-2026-06-05|AE5735|CNV     |CYYT  |EFRO       |active|2026-06-05 |2026-06-05 12:00:00|2026-06-05 12:20:00|0.0                |0.0              |0        |NULL         |2026-06-16 15:11:19.908289|
|EZD-92-2026-06-07  |758584|EZD92   |RPLL  |VMMC       |active|2026-06-07 |2026-06-07 18

## Couche Silver — Union flights_clean (2024) + flightaware_radar_clean  (2026)

### Architecture Lambda : fusion des données historiques et temps réel
- **flights_clean** : données historiques 2024 (batch )
- **flightaware_radar_clean** : données temps réel 2026 (AviationStack )
- La fusion aligne les schémas : chaque source remplit les colonnes qu'elle possède,
  les colonnes absentes reçoivent `lit(None)` avec le bon type.


In [12]:

# ────────────────────────────────────────────────────────────
# 1. NORMALISATION 2024 → Schéma commun strict
# ────────────────────────────────────────────────────────────


'''
Normalise les données 2024 vers un schéma commun. Les colonnes absentes des données historiques (comme icao24 ou latitude) sont 
remplies avec lit(None) casté au bon type pour garantir la compatibilité de schéma lors du union.
'''

flights_normalized = flights_clean.select(
    col("FlightDate").alias("flight_date"),
    col("IATA_Code_Marketing_Airline").cast(StringType()).alias("airline_code"),
    col("Flight_Number_Marketing_Airline").cast(StringType()).alias("flight_number"),
    col("Origin").alias("origin"),                      
    col("Dest").alias("dest"),                          
    col("OriginAirportID").cast(IntegerType()).alias("origin_airport_id"),
    col("DestAirportID").cast(IntegerType()).alias("dest_airport_id"),
    col("DepDelayMinutes").cast(DoubleType()).alias("departure_delay"),
    col("ArrDelayMinutes").cast(DoubleType()).alias("arrival_delay"),
    col("Cancelled").cast(IntegerType()).alias("cancelled"), # Force l'alignement en Integer
    col("Distance").cast(DoubleType()).alias("distance"),
    lit(None).cast(StringType()).alias("icao24"),
    lit(None).cast(StringType()).alias("callsign"),
    lit(None).cast(StringType()).alias("origin_country"),
    lit(None).cast(DoubleType()).alias("latitude"),
    lit(None).cast(DoubleType()).alias("longitude"),
    lit(None).cast(DoubleType()).alias("altitude_m"),
    lit(None).cast(DoubleType()).alias("speed_ms"),
    lit(None).cast(DoubleType()).alias("vertical_rate"),
    lit("historical").alias("source")
)

# ────────────────────────────────────────────────────────────
# 2. NORMALISATION 2026 (AviationStack) → Schéma commun strict
# ────────────────────────────────────────────────────────────


'''
Normalise les données 2026. airline_code est extrait du fa_flight_id (format ICAO-numéro-date).
Les colonnes absentes du temps réel (comme origin_airport_id) sont lit(None).
'''

flightaware_normalized = flightaware_radar_clean.select(
    col("flight_date"),
    
    # Sécurisation du split au cas où fa_flight_id est manquant ou mal formé
    when(col("fa_flight_id").contains("-"), split(col("fa_flight_id"), "-")[0])
        .otherwise(lit("Unknown")).cast(StringType()).alias("airline_code"),
    when(col("fa_flight_id").contains("-"), split(col("fa_flight_id"), "-")[1])
        .otherwise(lit("0000")).cast(StringType()).alias("flight_number"),
    
    col("origin"),                                      
    col("destination").alias("dest"),                   
    
    lit(None).cast(IntegerType()).alias("origin_airport_id"),
    lit(None).cast(IntegerType()).alias("dest_airport_id"),
    
    col("departure_delay_min").cast(DoubleType()).alias("departure_delay"),
    col("arrival_delay_min").cast(DoubleType()).alias("arrival_delay"),
    col("cancelled").cast(IntegerType()).alias("cancelled"), # Aligné
    lit(None).cast(DoubleType()).alias("distance"),
    col("icao24").cast(StringType()),
    col("callsign").cast(StringType()),
    lit("USA").cast(StringType()).alias("origin_country"),
    
    lit(None).cast(DoubleType()).alias("latitude"),
    lit(None).cast(DoubleType()).alias("longitude"),
    lit(None).cast(DoubleType()).alias("altitude_m"),
    lit(None).cast(DoubleType()).alias("speed_ms"),
    lit(None).cast(DoubleType()).alias("vertical_rate"),
    lit("realtime").alias("source")
)

# ────────────────────────────────────────────────────────────
# 3. ARCHITECTURE LAMBDA : UNION AVEC ALIGNEMENT STRICT DE SCHÉMA
# ────────────────────────────────────────────────────────────
vols_2024 = flights_normalized

try:
    vols_2026 = flightaware_normalized
    # Utilisation de la sélection explicite des colonnes de vols_2024 pour garantir l'ordre et le type lors du union
    vols_2026 = vols_2026.select(vols_2024.columns)
    
    vols_2026_count = vols_2026.count()

    if vols_2026_count > 0:
        flights_Radar = vols_2024.union(vols_2026)
    else:
        print("Aucune donnée temps réel, utilisation de l'historique seul.")
        flights_Radar = vols_2024
except Exception as e:
    print(f"Flux temps réel non disponible ({e}), utilisation de l'historique seul.")
    flights_Radar = vols_2024

flights_Radar = flights_Radar.withColumn("load_date", current_date())

print("✔ L'union ok ")

# Sauvegarde brute 

'''
Sauvegarde partitionnée par load_date avec compression Snappy (rapide, bonne compression).
repartition(20) équilibre la charge en 20 fichiers Parquet.
'''
flights_Radar \
    .repartition(20) \
    .write \
    .mode("append") \
    .option("compression", "snappy") \
    .partitionBy("load_date") \
    .parquet("s3a://data-lakehouse1/silver/flightaware_radar/")

# ────────────────────────────────────────────────────────────
# 4. Jointures et enrichissement
# ────────────────────────────────────────────────────────────


# ------------------------------------------------------------
# 1. Préparation des Lookups d'Aéroports (Origine et Destination)
# ------------------------------------------------------------
origin_lookup = broadcast(
    airports_clean.select(
        col("id").alias("origin_airport_id"),
        col("iata_code").alias("origin_iata_code"),
        col("name").alias("origin_airport_name"),
        col("municipality").alias("origin_city"),
        col("iso_country").alias("origin_country_airport")
    )
)

dest_lookup = broadcast(
    airports_clean.select(
        col("id").alias("dest_airport_id"),
        col("iata_code").alias("dest_iata_code"),
        col("name").alias("dest_airport_name"),
        col("municipality").alias("dest_city"),
        col("iso_country").alias("dest_country")
    )
)

# ------------------------------------------------------------
# 2. Application des Jointures d'Aéroports sur la table unifiée
# ------------------------------------------------------------

flights_Radar_airpOrigin = flights_Radar.join(
    origin_lookup,
    "origin_airport_id",
    "left"
)

flights_Radar_airport = flights_Radar_airpOrigin.join(
    dest_lookup,
    "dest_airport_id",
    "left"
)

# ------------------------------------------------------------
# 3. Join Météo sur la Date
# ------------------------------------------------------------

weather_broadcast = broadcast(weather_clean)

join_condition_weather = (
    (flights_Radar_airport["flight_date"] == weather_broadcast["Date"]) 
)

flightsRadar_enriched = flights_Radar_airport.join(
    weather_broadcast,
    join_condition_weather,
    "left"
).drop(weather_broadcast["Date"])

# ------------------------------------------------------------
# 4. Ajout du Timestamp d'Ingestion
# ------------------------------------------------------------
if "ingestion_date" in flightsRadar_enriched.columns:
    flightsRadar_enriched = flightsRadar_enriched.drop("ingestion_date")

flightsRadar_enriched = flightsRadar_enriched.withColumn(
    "ingestion_date",
    current_timestamp()
)

# Stockage finale
flightsRadar_enriched.write \
    .mode("overwrite") \
    .parquet("s3a://data-lakehouse1/silver/flightsRadar_enriched/")

print("✔ Sauvegarde vers MinIO ok ")

✔ L'union ok 
✔ Sauvegarde vers MinIO ok 


 Génération des logs opérationnels

In [13]:

# Échantillonnage sécurisé des vols depuis la couche Silver
flights_sample = (
    flightsRadar_enriched
    .select("flight_date", "origin", "dest", "flight_number")
    .limit(1000) # Collecte 1000 vols depuis Silver comme base pour générer des logs 
    .collect()
)


events = ["embarquement", "départ", "arrivée", "retard", "annulation", "changement_porte"]

data = []
# Génère 500 événements opérationnels aléatoires (embarquement, départ, retard, etc.) avec un timestamp réparti sur les 24h de la journée du vol.
for i in range(500):
    row = random.choice(flights_sample)
    flight_date = row["flight_date"]
    
    if flight_date is None:
        continue
        
    # SÉCURISATION : Conversion de datetime.date en datetime.datetime pour conserver l'heure du timedelta
    if not isinstance(flight_date, datetime):
        base_datetime = datetime.combine(flight_date, datetime.min.time())
    else:
        base_datetime = flight_date

    # Génération d'un timestamp réaliste réparti sur les 24h de la journée du vol
    event_timestamp = base_datetime + timedelta(minutes=random.randint(0, 1440))

    data.append({
        "log_id":     i + 1,
        "date_vol":   flight_date,
        "origin":     row["origin"],
        "dest":       row["dest"],
        "numero_vol": row["flight_number"],
        "event":      random.choice(events),
        "timestamp":  event_timestamp
    })

#  Création et ordonnancement du DataFrame Pandas
df_logs = pd.DataFrame(data)[["log_id", "date_vol", "origin", "dest", "numero_vol", "event", "timestamp"]]
#print("Logs générés :", len(df_logs))

# Sérialise les logs en CSV en mémoire (sans fichier temporaire) et les pousse directement dans MinIO via l'API S3.
csv_buffer = io.BytesIO()
df_logs.to_csv(csv_buffer, index=False, encoding='utf-8')
csv_buffer.seek(0)

# Connexion au stockage objet MinIO
s3 = boto3.client(
    "s3",
    endpoint_url="http://127.0.0.1:9000",  # 127.0.0.1 est souvent plus stable que localhost sous Windows/Docker
    aws_access_key_id="minioadmin",
    aws_secret_access_key="minioadmin"
)

# Injection directe dans le bucket data-lakehouse1 (Couche Bronze)
s3.put_object(
    Bucket="data-lakehouse1",
    Key="bronze/logs.csv",
    Body=csv_buffer.getvalue()
)

print("✔ Logs générés et sauvegardés avec succès dans MinIO (bronze/logs.csv)")

✔ Logs générés et sauvegardés avec succès dans MinIO (bronze/logs.csv)


## Phase 3 — Couche Gold (modèle en étoile)

In [14]:

# ------------------------------------------------------------
# Lecture Silver
# ------------------------------------------------------------
silver = spark.read.parquet(
    "s3a://data-lakehouse1/silver/flightsRadar_enriched/"
)

# ------------------------------------------------------------
# dim_airport
# ------------------------------------------------------------
dim_airport = (
    airports_clean
    .dropDuplicates(["id"])
    .filter(col("id").isNotNull())
    .withColumnRenamed("id",           "airport_id")
    .withColumnRenamed("municipality", "city")
    .withColumnRenamed("iso_country",  "country")
    .withColumnRenamed("latitude_deg", "latitude")
    .withColumnRenamed("longitude_deg","longitude")
    .select("airport_id", "iata_code", "name", "city", "country", "latitude", "longitude")
)


# ------------------------------------------------------------
# dim_time
# ------------------------------------------------------------
window_time = Window.orderBy("date")

dim_time = (
    silver
    .select("flight_date").dropDuplicates()
    .withColumnRenamed("flight_date", "date")
    .withColumn("date",        to_date(col("date")))
    .withColumn("day",         dayofmonth("date"))
    .withColumn("month",       month("date"))
    .withColumn("year",        year("date"))
    .withColumn("day_of_week", dayofweek("date"))
    .withColumn("is_weekend",  when(dayofweek("date").isin(1, 7), 1).otherwise(0))
    .withColumn("time_id",     row_number().over(window_time))
    .select("time_id", "date", "day", "month", "year", "day_of_week", "is_weekend")
)


# ------------------------------------------------------------
# dim_weather
# ------------------------------------------------------------
window_weather = Window.orderBy("date")

dim_weather = (
    silver
    .filter(col("Temperature_C").isNotNull())   # garder seulement les lignes avec météo 
    .filter(col("condition").isNotNull())  # supprimer les lignes où condition est NULL
    .select(
        col("flight_date").alias("date"),
        col("Temperature_C").alias("temperature"),
        col("Wind_Speed_km_h").alias("wind_speed"),
        col("Precipitation_mm").alias("precipitation"),
        col("Condition").alias("condition")
    )
    .dropDuplicates(["date"])
    .withColumn("weather_id", row_number().over(window_weather))
    .select("weather_id", "date", "temperature", "wind_speed", "precipitation", "condition")
)

# ------------------------------------------------------------
# fact_flights
# ------------------------------------------------------------
fact_base = (
    silver
    .select(
        "origin_airport_id",
        "dest_airport_id",
        col("flight_date").alias("date"),
        "airline_code",
        "departure_delay",
        "arrival_delay",
        "cancelled",
        "icao24",
        "latitude",
        "longitude",
        "altitude_m",
        "speed_ms",
        "vertical_rate",
        "source"
    )
)



fact_flights = fact_base.join(
    dim_time.select("date", "time_id"),
    on="date",
    how="left"
)

fact_flights = fact_flights.join(
    dim_weather.select("date", "weather_id"),
    on="date",
    how="left"
)

# Joint la table de faits avec les clés des dimensions. coalesce(..., lit(-1)) : les vols sans correspondance météo reçoivent weather_id = -1 (sentinel row) au lieu de NULL.

fact_flights = fact_flights.withColumn(
    "weather_id",
    coalesce(col("weather_id"), lit(-1))
)
 
from pyspark.sql.functions import row_number, lit
from pyspark.sql import Window

# Forcer 1 seule partition → row_number() sera vraiment global de 1 à N
fact_flights = fact_flights.coalesce(1)

window_fact = Window.orderBy("date", "airline_code")

fact_flights = (
    fact_flights
    .withColumn("flight_id", row_number().over(window_fact))
    .select(
        "flight_id",
        "origin_airport_id",
        "dest_airport_id",
        "time_id",
        "weather_id",
        "airline_code",
        "departure_delay",
        "arrival_delay",
        "cancelled",
    )
)


# ------------------------------------------------------------
# Sauvegarde dans Gold
# ------------------------------------------------------------
dim_airport.write.mode("overwrite").parquet(
    "s3a://data-lakehouse1/gold/dim_airport/"
)

dim_time.write.mode("overwrite").parquet(
    "s3a://data-lakehouse1/gold/dim_time/"
)

dim_weather.write.mode("overwrite").parquet(
    "s3a://data-lakehouse1/gold/dim_weather/"
)

fact_flights.repartition(20).write.mode("overwrite") \
    .option("compression", "snappy") \
    .parquet("s3a://data-lakehouse1/gold/fact_flights/")

print("✔ Sauvegarde dans Gold vers MinIO terminée")



✔ Sauvegarde dans Gold vers MinIO terminée


## Phase 4 — Chargement PostgreSQL

In [17]:
# ── 1. Paramètres de connexion ────────────────────────────────────────────────
DB_USER = "postgres"
DB_PASS = "mary"
DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "lakehouse_dbs"

DB_URL = f"postgresql+psycopg2://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
#  Crée le moteur SQLAlchemy pour PostgreSQL.
engine = create_engine(DB_URL, connect_args={"connect_timeout": 10})

# ── 2. Tester la connexion ────────────────────────────────────────────────────
try:
    with engine.connect() as conn:
        v = conn.execute(text("SELECT version()")).fetchone()[0]
        print("Connexion PostgreSQL OK")
except Exception as e:
    print("ERREUR de connexion :", e)
    raise

# ── 3. MinIO ───────────────────────────────────────────────────────────
s3 = boto3.client(
    "s3",
    endpoint_url="http://localhost:9000",
    aws_access_key_id="minioadmin",
    aws_secret_access_key="minioadmin",
)
BUCKET = "data-lakehouse1"

# Lit tous les fichiers Parquet d'un préfixe MinIO en mémoire (via BytesIO) et les concatène en un seul DataFrame Pandas.
def read_parquet_from_minio(prefix):
    resp = s3.list_objects_v2(Bucket=BUCKET, Prefix=prefix)
    files = [obj["Key"] for obj in resp.get("Contents", [])
             if obj["Key"].endswith(".parquet")]
    if not files:
        raise FileNotFoundError(f"Aucun fichier sous s3://{BUCKET}/{prefix}")
    frames = []
    for key in files:
        buf = io.BytesIO(s3.get_object(Bucket=BUCKET, Key=key)["Body"].read())
        frames.append(pq.read_table(buf).to_pandas())
    return pd.concat(frames, ignore_index=True)



# Stratégie de chargement en deux étapes :

#  - df.head(0).to_sql(...) : crée la table vide avec le bon schéma via SQLAlchemy (DDL automatique)
#  - COPY FROM STDIN : charge les données via le protocole COPY de PostgreSQL (beaucoup plus rapide que des INSERT individuels pour des millions de lignes)

# na_rep="\\N" : représente les valeurs NULL en CSV par \N, reconnu par PostgreSQL comme NULL.


def load_with_copy(df, table_name):
    # Étape 1 : créer la table vide via SQLAlchemy
        df.head(0).to_sql(table_name, engine, if_exists="replace", index=False)

        # Étape 1.5 : Si c'est la table de faits, on force flight_id en Clé Primaire
        if table_name == "fact_flights":
            with engine.connect() as conn:
                conn.execute(text("ALTER TABLE fact_flights ADD PRIMARY KEY (flight_id);"))
                conn.commit()

        # Étape 2 : insérer les données via COPY
        conn = psycopg2.connect(
            dbname=DB_NAME, user=DB_USER, password=DB_PASS,
            host=DB_HOST, port=DB_PORT
        )
        cur = conn.cursor()
        buf = io.StringIO()
        df.to_csv(buf, index=False, header=False, na_rep="\\N", quoting=csv.QUOTE_MINIMAL)
        buf.seek(0)
        cur.copy_expert(
            f"COPY {table_name} FROM STDIN WITH (FORMAT CSV, NULL '\\N')",
            buf
        )
        conn.commit()
        cur.close()
        conn.close()


# ── 4. Charger les tables depuis Gold ────────────────────────────────────────
GOLD_TABLES = [
    ("dim_airport",  "gold/dim_airport/"),
    ("dim_time",     "gold/dim_time/"),
    ("dim_weather",  "gold/dim_weather/"),
    ("fact_flights", "gold/fact_flights/"),
]


for table, prefix in GOLD_TABLES:
    df = read_parquet_from_minio(prefix)
    
    # Réassigner flight_id proprement
    if table == "fact_flights":
        df = df.sort_values(["time_id", "airline_code"]).reset_index(drop=True)
        df["flight_id"] = df.index + 1  # commence à 1
    
    load_with_copy(df, table)


# ── 5. Créer les index ────────────────────────────────────────────────────────

# Crée 5 index sur fact_flights après le chargement (plus rapide que pendant). Ces index accélèrent les jointures de Power BI sur les clés étrangères.
with engine.connect() as conn:
    conn.execute(text("CREATE INDEX IF NOT EXISTS idx_ff_origin  ON fact_flights(origin_airport_id)"))
    conn.execute(text("CREATE INDEX IF NOT EXISTS idx_ff_dest    ON fact_flights(dest_airport_id)"))
    conn.execute(text("CREATE INDEX IF NOT EXISTS idx_ff_time    ON fact_flights(time_id)"))
    conn.execute(text("CREATE INDEX IF NOT EXISTS idx_ff_weather ON fact_flights(weather_id)"))
    conn.execute(text("CREATE INDEX IF NOT EXISTS idx_ff_airline   ON fact_flights(airline_code)"))
    conn.commit()

print("\n  Chargement dans PostgreSQL terminé ")

Connexion PostgreSQL OK

  Chargement dans PostgreSQL terminé 


# CONCLUSION

En résumé, notre projet implémente une architecture Data Lakehouse de bout en bout :

- Ingestion batch (CSV, JSON) et temps réel (Kafka + AviationStack API)
- Pipeline Bronze → Silver → Gold 
- Union de données historiques 2024 et temps réel 2026 dans un schéma unifié (Architecture Lambda)
- Modèle en étoile prêt pour l'analyse BI dans PostgreSQL


# Améliorations : 

 - utilise un format de table ouvert (Open Table Format) comme Apache Iceberg, Delta Lake ou Apache Hudi à la place de fichiers CSV ou Parquet.
 - Utilisez une stratégie d'Upsert (Insert or Update via SQLAlchemy ou des tables de staging temporaires dans PostgreSQL) pour mettre à jour uniquement les lignes modifiées sans perturber le reste de la base.
